In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
!pip install -U  git+https://github.com/huggingface/transformers.git
!pip install -U  git+https://github.com/huggingface/accelerate.git
!pip install -U git+https://github.com/ytdl-org/youtube-dl.git

URLs = []

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-nqp98o6r
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-nqp98o6r
  Resolved https://github.com/huggingface/transformers.git to commit d211a84aca8a8513a599171173c29bc8d3061fba
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-4.31.0.dev0-py3-none-any.whl size=7283372 sha256=f0e2e3024616874a7b647f8a1a1aeae15d324b657b9d6cbe03796fa9915c42cd
  Stored in directory: /tmp/pip-ephem-wheel-cache-c9fasv7e/wheels/e7/9c/5b/e1a9c8007c343041e61cc484433d512ea9274272e3fcbe7c16
Successfully built transformers
  Attempting uninstall: transformers
    Found existing installation: transformers 4.30.1
    Uninstalling transformers-4.30.1:
      Successfully uninstalled transformers-4.30.1
  Cloning https://github.com

In [2]:
from youtube_dl import YoutubeDL
import logging

LOGGER = logging.getLogger("verbose")
LOGGER.setLevel(logging.INFO)


def download_from_urls(urls, opts={}):
    global LOGGER
    LOGGER.info("cleaning up dir")
    print("cleaning up dir")
    [
        os.remove(file) for file in filter(
            lambda filename: filename.endswith(".mp3"), os.listdir())
    ]
    opts.update({
        'format':
        'bestaudio/best',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '192',
        }],
        'outtmpl':
        '%(id)s.%(ext)s'
    })
    with YoutubeDL(opts) as downloader:
        LOGGER.info("downloading videos")
        print("downloading videos")
        downloader.download(urls)

In [3]:
from transformers import pipeline
import torch

device = "cuda:0" if torch.cuda.is_available() else "cpu"
transcribe = pipeline(task="automatic-speech-recognition",
                      model="vasista22/whisper-telugu-medium",
                      chunk_length_s=30,
                      device=device)
transcribe.model.config.forced_decoder_ids = transcribe.tokenizer.get_decoder_prompt_ids(
    language="te", task="transcribe")

/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:98: UserWarning: unable to load libtensorflow_io_plugins.so: unable to open file: libtensorflow_io_plugins.so, from paths: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so']
caused by: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so: undefined symbol: _ZN3tsl6StatusC1EN10tensorflow5error4CodeESt17basic_string_viewIcSt11char_traitsIcEENS_14SourceLocationE']
  warnings.warn(f"unable to load libtensorflow_io_plugins.so: {e}")
/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:104: UserWarning: file system plugins are not loaded: unable to open file: libtensorflow_io.so, from paths: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io.so']
caused by: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io.so: undefined symbol: _ZTVN10tenso

In [4]:
import json

download_from_urls(URLs)

cleaning up dir
downloading videos
[youtube] vzSYXNp0XXk: Downloading webpage
[youtube] vzSYXNp0XXk: Downloading player 23604418
[dashsegments] Total fragments: 2
[download] Destination: vzSYXNp0XXk.webm
[download] 100% of 13.28MiB in 00:00.77MiB/s ETA 00:004
[ffmpeg] Destination: vzSYXNp0XXk.mp3
Deleting original file vzSYXNp0XXk.webm (pass -k to keep)
[youtube] lJ8dFeO8ZpY: Downloading webpage
[dashsegments] Total fragments: 2
[download] Destination: lJ8dFeO8ZpY.webm
[download] 100% of 13.05MiB in 00:00.12MiB/s ETA 00:003
[ffmpeg] Destination: lJ8dFeO8ZpY.mp3
Deleting original file lJ8dFeO8ZpY.webm (pass -k to keep)


In [5]:
with open('transcriptions.json', 'w') as json_file:
    trans_dict = {}
    for audio in filter(lambda filename: filename.endswith(".mp3"),
                        os.listdir()):
        LOGGER.info(f"Transcribing {audio}")
        print(f"Transcribing {audio}")
        trans_dict["https://youtu.be/" + audio.split(".")[0]] = transcribe(
            audio, num_workers=1)["text"]
    print(trans_dict)
    json.dump(trans_dict, json_file)

Transcribing lJ8dFeO8ZpY.mp3


ffmpeg: /opt/conda/lib/libncursesw.so.6: no version information available (required by /lib/x86_64-linux-gnu/libcaca.so.0)
ffmpeg: /opt/conda/lib/libncursesw.so.6: no version information available (required by /lib/x86_64-linux-gnu/libcaca.so.0)
/opt/conda/lib/python3.10/site-packages/transformers/generation/utils.py:1363: UserWarning: Using `max_length`'s default (448) to control the generation length. This behaviour is deprecated and will be removed from the config in v5 of Transformers -- we recommend using `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Transcribing vzSYXNp0XXk.mp3


ffmpeg: /opt/conda/lib/libncursesw.so.6: no version information available (required by /lib/x86_64-linux-gnu/libcaca.so.0)
ffmpeg: /opt/conda/lib/libncursesw.so.6: no version information available (required by /lib/x86_64-linux-gnu/libcaca.so.0)


{'https://youtu.be/lJ8dFeO8ZpY': 'ఏమి లేనప్పుడు అన్నీ ఉన్నట్లుగా ఉండండి అన్నీ ఉన్నప్పుడు ఏమి లేనట్లుగా ఉండండి ఇదే మనిషి మనుగడకు రహస్యం ఎదుటి మనిషికి నీ మీద ప్రేమ లేనప్పుడు నీవు ఎంత ఆరాటపడినా వ్యర్ధమే ఒక మనిషికి నిలబడటం తేలిక పరిగెత్తటం కష్టం మన మనసుకి పరిగెత్తటం తేలిక నిలబడటం కష్టం నీ బద్ధకమే నీ అసలు శత్రువు అదే నీ ఎదుగుదలకు అడ్డం నీ పేదరికానికి మూలకారణం ఎదురుగా ఉన్నది ఎవరైనా సరే ఎదిరించి నిలబడు అతనికి కావాల్సింది నీ మరణమైనా తెగించి నిలబడు గంధపు చెట్టు తనను నరికే గొడ్డలికి కూడా పరిమళాన్ని ఇస్తుంది అలాగే మంచి వ్యక్తిత్వం ఉన్నవాడు తనను దూషించే వారికి కూడా మంచే చేస్తాడు మీ దగ్గర ఎంత సంపద ఉన్నప్పటికీ మీరు శారీరకంగా ఎంత బలంగా ఉన్నప్పటికీ మానసిక ప్రశాంతే అన్ని వ్యర్థమే కష్టాలు రావడం కూడా ఒక విధంగా మంచిదే ఎందుకంటే కష్టాలు వస్తేనే కదా మన కన్నీళ్లు తుడవటానికి ఎవరున్నారో తెలిసేది మీరు ప్రశాంతంగా ఉండాలనుకుంటే అనవసరమైనవి అంటించుకోకండి పనికిరానివి పట్టించుకోకండి మన దగ్గర మనీ తీసుకునేటప్పుడు ఉండే మర్యాద మంచితనం తిరిగి ఇచ్చేటప్పుడు ఉండవు గెలవాలి అనుకున్నప్పుడు కష్టం మొదలవుతుంది ఎలాగైనా గెలవాలి అనుకున

In [ ]:
[
    os.remove(file) for file in filter(
        lambda filename: filename.endswith(".mp3"), os.listdir())
]